In [13]:
import heapq 
import csv
from collections import defaultdict
from itertools import combinations
import ipywidgets as widgets

In [ ]:
TIEMPO_TRANSBORDO = 4


def cargar_grafo(archivo):
    # Lista de adyajencia
    grafo = defaultdict(dict)
    estaciones_por_nombre = defaultdict(set)

    # Leer el csv
    with open(archivo, mode='r', encoding='utf-8-sig', newline='') as file:
        # Guardar las lineas en la lista de adyajencia
        for row in csv.DictReader(file):
            u = row['from_stop_id'].strip()
            v = row['to_stop_id'].strip()
            peso = float(row['mean_travel_time_min'])
            
            grafo[u][v] = peso
            grafo.setdefault(v, {})

            ruta = row['route_id'].strip()
            for estacion_id, columna in ((u, 'from_stop_name'), (v, 'to_stop_name')):
                nombre = ' '.join(row[columna].split()).casefold()
                if nombre:
                    estaciones_por_nombre[nombre].add((estacion_id, ruta))

    # Guardar aristas entre lineas
    for estaciones in estaciones_por_nombre.values():
        for (origen, ruta_origen), (destino, ruta_destino) in combinations(estaciones, 2):
            if ruta_origen == ruta_destino or origen == destino:
                continue
            grafo[origen].setdefault(destino, TIEMPO_TRANSBORDO)
            grafo[destino].setdefault(origen, TIEMPO_TRANSBORDO)

    return grafo


def dijkstra(grafo, inicio, final):
    tiempos = {nodo: float('inf') for nodo in grafo}
    tiempos[inicio] = 0
    anterior = {}

    heap = [(0, inicio)]

    while heap:
        tiempo_act, nodo_act = heapq.heappop(heap)

        if nodo_act == final:
            break

        if tiempo_act > tiempos[nodo_act]:
            continue
        
        for vecino, peso in grafo[nodo_act].items():
            tiempo = tiempo_act + peso

            if tiempo < tiempos[vecino]:
                tiempos[vecino] = tiempo
                anterior[vecino] = nodo_act
                heapq.heappush(heap, (tiempo, vecino))

    # Contruir el camino
    camino = []
    actual = final
    while actual in anterior or actual == inicio:
        camino.append(actual)
        if actual == inicio:
            break
        actual = anterior[actual]

    return tiempos[final], camino[::-1]



grafo = cargar_grafo('metro_cdmx_aristas.csv')

In [ ]:
nombres = {}

with open('metro_cdmx_aristas.csv', encoding='utf-8-sig', newline='') as file:
    for row in csv.DictReader(file):
        estacion_id = row['from_stop_id'].strip()
        nombre = row['from_stop_name'].strip()
        linea = row['route_short_name'].removeprefix('L')
        nombres[estacion_id] = f'{nombre} (Línea {linea})'

opciones = sorted((nombre, estacion_id) for estacion_id, nombre in nombres.items())

origen = widgets.Dropdown(
    options=opciones, description='Inicio'
)
destino = widgets.Dropdown(
    options=opciones, description='Final'
)

def mostrar_ruta(inicio, final):
    tiempo, camino = dijkstra(grafo, inicio, final)

    if not camino:
        print('No hay una ruta entre las estaciones seleccionadas.')
        return

    print(f'Tiempo minimo: {tiempo:.1f} minutos')
    print(f'Ruta:')
    print('\n↓\n'.join(nombres[estacion_id] for estacion_id in camino))

salida = widgets.interactive_output(
    mostrar_ruta, {'inicio': origen, 'final': destino}
)

display(widgets.VBox([origen, destino, salida]))